**Smart Energy Analytics**

In [ ]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
uploaded = files.upload()
spark = SparkSession.builder \
    .appName("EnergyAssessment") \
    .getOrCreate()
def load_csv(filename):
    return spark.read.csv(
        filename,
        header=True,
        inferSchema=True
    )
readings_df = load_csv("meter_readings_90000.csv")
extra_df = load_csv("meter_readings_extra_12000.csv")
meter_df = load_csv("meter_master.csv")
substation_df = load_csv("substation_master.csv")
tariff_df = load_csv("tariff_master.csv")
maintenance_df = load_csv("maintenance_tickets_1600.csv")

Saving tariff_master.csv to tariff_master.csv
Saving substation_master.csv to substation_master.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
Saving meter_master.csv to meter_master.csv


SECTION 01 / DATA PREPARATION & ANALYSIS

A. Inspect, select and transform

In [ ]:
readings_df.printSchema()
print("Count:",readings_df.count())
readings_df.show(10)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Count: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 

In [ ]:
readings_df.select(
    "reading_id",
    "meter_id",
    "reading_timestamp",
    "energy_kwh",
    "reading_status"
    ).show()
renamed_df=readings_df.withColumnRenamed("source_type","collection_mode")
renamed_df.show()

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
| RD0000006|   M0139|2026-02-20 02:15:00|      0.36|         Valid|
| RD0000007|   M0142|2026-02-01 05:30:00|     79.47|         Valid|
| RD0000008|   M0126|2026-03-01 20:45:00|     28.35|      Rejected|
| RD0000009|   M0005|2026-05-03 10:30:00|      1.96|     Estimated|
| RD0000010|   M0052|2026-04-10 05:45:00|     22.52|         Valid|
| RD0000011|   M0137|2026-02-03 16:00:00|     10.51|         Valid|
| RD0000012|   M0009|2026-05-01 00:00:00|      1

In [ ]:
from pyspark.sql.functions import col, when

In [ ]:
readings_df.filter(
    (col("reading_status")=="Valid")&(col("energy_kwh")>20)
).sort(col("energy_kwh").desc()).show()

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             0|         Valid| SmartMeter|                 25.3|      OP017|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             

In [ ]:
band_df=readings_df.withColumn(
    "energy_band",
    when(col("energy_kwh").isNull(),"Unknown")
    .when(col("energy_kwh")<5,"Low")
    .when(col("energy_kwh")<25,"Medium")
    .when(col("energy_kwh")>25,"High")
)
band_df.show()

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|energy_band|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid| SmartMeter|                 36.2|      OP029|     Medium|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|        Low|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|     Medium|
| RD0000004|   M

B. Null values and data cleaning

In [ ]:
readings_df.select(
    F.sum(col("energy_kwh").isNull().cast("int")).alias("energy_nulls"),
    F.sum(col("voltage_v").isNull().cast("int")).alias("voltage_nulls"),
    F.sum(col("power_factor").isNull().cast("int")).alias("power_nulls"),
    F.sum(col("operator_id").isNull().cast("int")).alias("operator_nulls")
).show()

+------------+-------------+-----------+--------------+
|energy_nulls|voltage_nulls|power_nulls|operator_nulls|
+------------+-------------+-----------+--------------+
|        1620|         2070|       1440|          5400|
+------------+-------------+-----------+--------------+



In [ ]:
filtered_df=readings_df.fillna({
    "operator_id":"UNASSIGNED",
    "power_factor":0.90
})
filtered_df.select("operator_id","power_factor").show()

+-----------+------------+
|operator_id|power_factor|
+-----------+------------+
|      OP029|       0.939|
|      OP025|       0.845|
|      OP007|       0.879|
| UNASSIGNED|       0.831|
|      OP019|       0.966|
|      OP021|       0.788|
|      OP016|       0.986|
|      OP025|       0.818|
|      OP010|       0.817|
|      OP028|       0.804|
|      OP030|         0.9|
|      OP027|        0.81|
|      OP005|       0.807|
|      OP019|       0.884|
|      OP036|       0.919|
|      OP035|       0.895|
|      OP003|       0.762|
|      OP021|       0.914|
|      OP014|       0.953|
|      OP020|       0.771|
+-----------+------------+
only showing top 20 rows


In [ ]:
before=readings_df.count()
filtered_df=readings_df.dropna(
    subset=["energy_kwh","voltage_v"]
)
after=filtered_df.count()
print("Removed:",before-after)

Removed: 3651


C. Joins, groupBy and aggregations

In [ ]:
joined_df=readings_df.join(
    meter_df,
    "meter_id"
)
joined_df.filter(
    col("reading_status")=="Valid"
).groupBy("site_type").agg(
    F.sum("energy_kwh").alias("total_energy")
).show()

+---------------+-----------------+
|      site_type|     total_energy|
+---------------+-----------------+
|Public Facility|        138870.41|
|    Residential|100863.2900000004|
|     Industrial|710317.4100000021|
|     Commercial|451525.1799999996|
+---------------+-----------------+



In [ ]:
region_df=joined_df.join(
    substation_df,
    "substation_id"
)
region_df.filter(
    col("reading_status")=="Valid"
).groupBy("region","city").agg(
    F.sum("energy_kwh").alias("total_energy")
).sort("region","city").show()

+-------+-------------+------------------+
| region|         city|      total_energy|
+-------+-------------+------------------+
|Central|       Indore|176503.91000000012|
|  North|        Delhi| 94234.89000000014|
|  North|       Jaipur| 47863.15999999996|
|  North|      Lucknow| 142610.3999999998|
|  South|    Bengaluru| 73787.59999999998|
|  South|      Chennai| 51829.05999999996|
|  South|    Hyderabad|51937.789999999884|
|  South|        Kochi| 204272.4600000002|
|  South|Visakhapatnam|205015.82999999932|
|   West|    Ahmedabad| 138627.7400000004|
|   West|       Mumbai| 52116.20999999999|
|   West|         Pune|162777.23999999996|
+-------+-------------+------------------+



In [ ]:
joined_df.join(
    tariff_df,
    "tariff_id"
).filter(
    (col("reading_status")!="Rejected")&
    col("energy_kwh").isNotNull()
).withColumn(
    "energy_charge_inr",
    F.col("energy_kwh") * F.col("rate_per_kwh_inr")
).show()

+-------------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-------------+---------------+-----------------+-----------------+------------+--------------------+---------------+----------------+------------------------+------------------+
|          tariff_id|meter_id|reading_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|substation_id|      site_type|installation_date|rated_capacity_kw|meter_status|         tariff_name|  site_category|rate_per_kwh_inr|monthly_fixed_charge_inr| energy_charge_inr|
+-------------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-------------+---------------+-----------------+-----------------+------------+--------------------+---------------+---------------

In [ ]:
meter_df.join(
    maintenance_df,
    "meter_id"
).groupBy("site_type","severity").agg(
    F.count("*").alias("total_tickets"),
    F.avg("maintenance_cost_inr").alias("avg_maintenance_cost")
).show()

+---------------+--------+-------------+--------------------+
|      site_type|severity|total_tickets|avg_maintenance_cost|
+---------------+--------+-------------+--------------------+
|    Residential|Critical|           41|  11875.782702702705|
|     Industrial|  Medium|          106|  12043.898526315788|
|     Commercial|Critical|           37|  13424.726216216219|
|     Commercial|     Low|          166|  12223.517692307689|
|Public Facility|    High|           48|  12905.279347826086|
|    Residential|     Low|          194|   12531.90201117318|
|Public Facility|  Medium|           97|  13050.628421052634|
|     Industrial|Critical|           17|  12296.198000000002|
|Public Facility|Critical|           15|   8986.843571428572|
|     Industrial|    High|           63|  12838.869344262299|
|    Residential|    High|          156|  11047.967122302161|
|Public Facility|     Low|           60|  11195.929464285715|
|     Commercial|  Medium|          171|  13706.600886075952|
|     Co

In [ ]:
region_df.groupBy("substation_id").agg(
    F.avg("voltage_v").alias("avg_voltage"),
    F.sum("outage_minutes").alias("total outage")
).sort(col("total outage").desc()).show(5)

+-------------+------------------+------------+
|substation_id|       avg_voltage|total outage|
+-------------+------------------+------------+
|        SUB10|230.04010567581344|       91224|
|        SUB09|230.04586272640606|       81734|
|        SUB04| 230.0558687466385|       75872|
|        SUB08|229.88012557077673|       75581|
|        SUB03|230.03094896227708|       61662|
+-------------+------------------+------------+
only showing top 5 rows


In [ ]:
readings_df.groupBy("source_type","reading_status").agg(
     F.count("*").alias("reading_count")
).orderBy(F.col("reading_count").desc()).show()

+-----------+--------------+-------------+
|source_type|reading_status|reading_count|
+-----------+--------------+-------------+
| SmartMeter|         Valid|        65521|
| SmartMeter|     Estimated|        10443|
|     Manual|         Valid|         8965|
| SmartMeter|      Rejected|         3202|
|     Manual|     Estimated|         1405|
|     Manual|      Rejected|          464|
+-----------+--------------+-------------+



SECTION 02 / ADVANCED DATAFRAME OPERATIONS

D. Date and window functions

In [ ]:
date_df = readings_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
).withColumn(
    "year",
    F.year("reading_timestamp")
).withColumn(
    "month",
    F.month("reading_timestamp")
)
monthly_df = date_df.join(
    meter_df, "meter_id"
).join(
    substation_df, "substation_id"
)
monthly_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "region", "year", "month"
).agg(
    F.sum("energy_kwh").alias("monthly_energy")
).orderBy(
    "region", "year", "month"
).show()

+-------+----+-----+------------------+
| region|year|month|    monthly_energy|
+-------+----+-----+------------------+
|Central|2026|    1| 31475.57999999999|
|Central|2026|    2|28161.410000000007|
|Central|2026|    3|          29805.85|
|Central|2026|    4|29475.339999999982|
|Central|2026|    5|30048.879999999986|
|Central|2026|    6| 27536.84999999999|
|  North|2026|    1|48849.309999999925|
|  North|2026|    2|45444.249999999985|
|  North|2026|    3| 51094.61000000006|
|  North|2026|    4| 43451.02000000005|
|  North|2026|    5| 48796.93000000004|
|  North|2026|    6| 47072.32999999992|
|  South|2026|    1| 97547.38999999987|
|  South|2026|    2| 90864.80000000006|
|  South|2026|    3|102258.77000000005|
|  South|2026|    4| 97266.94000000005|
|  South|2026|    5| 101204.8699999999|
|  South|2026|    6| 97699.97000000002|
|   West|2026|    1| 59967.56999999997|
|   West|2026|    2| 55807.89999999989|
+-------+----+-----+------------------+
only showing top 20 rows


In [ ]:
meter_energy = joined_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "substation_id", "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy")
)
w = Window.partitionBy(
    "substation_id"
).orderBy(
    F.col("total_energy").desc()
)
ranked_df = meter_energy.withColumn(
    "rank",
    F.dense_rank().over(w)
)
ranked_df.filter(
    F.col("rank") <= 3
).orderBy(
    "substation_id", "rank"
).show(50)

+-------------+--------+------------------+----+
|substation_id|meter_id|      total_energy|rank|
+-------------+--------+------------------+----+
|        SUB01|   M0130|           9364.98|   1|
|        SUB01|   M0110| 9259.750000000002|   2|
|        SUB01|   M0068| 8832.580000000005|   3|
|        SUB02|   M0071|          29242.04|   1|
|        SUB02|   M0145|28641.319999999992|   2|
|        SUB02|   M0133| 7060.140000000003|   3|
|        SUB03|   M0096| 9632.740000000003|   1|
|        SUB03|   M0080| 9525.940000000002|   2|
|        SUB03|   M0073| 9369.789999999995|   3|
|        SUB04|   M0038|          29505.88|   1|
|        SUB04|   M0099|29216.319999999996|   2|
|        SUB04|   M0098|28613.460000000014|   3|
|        SUB05|   M0094| 9659.639999999996|   1|
|        SUB05|   M0050| 9570.859999999997|   2|
|        SUB05|   M0008| 9524.780000000004|   3|
|        SUB06|   M0126|32573.380000000023|   1|
|        SUB06|   M0128|30748.840000000004|   2|
|        SUB06|   M0

In [ ]:
w = Window.partitionBy(
    "meter_id"
).orderBy(
    "reading_timestamp",
    "reading_id"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)
cumulative_df = date_df.filter(
    F.col("reading_status") == "Valid"
).withColumn(
    "cumulative_energy",
    F.sum("energy_kwh").over(w)
)
cumulative_df.select(
    "meter_id",
    "reading_timestamp",
    "energy_kwh",
    "cumulative_energy"
).show(20)

+--------+-------------------+----------+------------------+
|meter_id|  reading_timestamp|energy_kwh| cumulative_energy|
+--------+-------------------+----------+------------------+
|   M0001|2026-01-01 01:15:00|     22.27|             22.27|
|   M0001|2026-01-01 03:30:00|      4.05|             26.32|
|   M0001|2026-01-01 09:45:00|     11.28|              37.6|
|   M0001|2026-01-02 04:45:00|     11.76|             49.36|
|   M0001|2026-01-02 08:15:00|     10.06|             59.42|
|   M0001|2026-01-02 18:30:00|     15.83|             75.25|
|   M0001|2026-01-03 00:30:00|     17.06|             92.31|
|   M0001|2026-01-03 06:00:00|     17.14|            109.45|
|   M0001|2026-01-03 08:30:00|     20.46|            129.91|
|   M0001|2026-01-03 18:15:00|      20.4|            150.31|
|   M0001|2026-01-04 19:30:00|      6.84|            157.15|
|   M0001|2026-01-05 10:30:00|     10.62|            167.77|
|   M0001|2026-01-05 11:15:00|     12.52|180.29000000000002|
|   M0001|2026-01-05 17:

E. Union and deduplication

In [ ]:
combined_df = readings_df.unionByName(extra_df)
print("Main:", readings_df.count())
print("Extra:", extra_df.count())
print("Combined:", combined_df.count())

Main: 90000
Extra: 12000
Combined: 102000


In [ ]:
duplicates_df = combined_df.groupBy(
    "reading_id"
).count().filter(
    F.col("count") > 1
)
duplicates_df.show()
unique_df = combined_df.dropDuplicates(
    ["reading_id"]
)
print("Unique count:", unique_df.count())

+----------+-----+
|reading_id|count|
+----------+-----+
| RD0025848|    2|
| RD0045533|    2|
| RD0064921|    2|
| RD0056575|    2|
| RD0058815|    2|
| RD0022131|    2|
| RD0029598|    2|
| RD0045710|    2|
| RD0030877|    2|
| RD0002679|    2|
| RD0011169|    2|
| RD0028736|    2|
| RD0030669|    2|
| RD0061933|    2|
| RD0015904|    2|
| RD0008188|    2|
| RD0037899|    2|
| RD0000004|    2|
| RD0025628|    2|
| RD0048911|    2|
+----------+-----+
only showing top 20 rows
Unique count: 101300


F. Lazy execution, caching and persistence

In [ ]:
pipeline_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).withColumn(
    "double_energy",
    F.col("energy_kwh") * 2
).groupBy(
    "meter_id"
).agg(
    F.sum("double_energy").alias("total_double_energy")
).orderBy(
    F.col("total_double_energy").desc()
)
pipeline_df.explain()
pipeline_df.show(10)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_double_energy#1023 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_double_energy#1023 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2234]
      +- HashAggregate(keys=[meter_id#18], functions=[sum(double_energy#1022)])
         +- Exchange hashpartitioning(meter_id#18, 200), ENSURE_REQUIREMENTS, [plan_id=2231]
            +- HashAggregate(keys=[meter_id#18], functions=[partial_sum(double_energy#1022)])
               +- Project [meter_id#18, (energy_kwh#20 * 2.0) AS double_energy#1022]
                  +- Filter (isnotnull(reading_status#24) AND (reading_status#24 = Valid))
                     +- FileScan csv [meter_id#18,energy_kwh#20,reading_status#24] Batched: false, DataFilters: [isnotnull(reading_status#24), (reading_status#24 = Valid)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(reading_stat

In [ ]:
import time

In [ ]:
valid_df = readings_df.filter(
    F.col("reading_status") == "Valid"
)
valid_df.cache()
start = time.time()
valid_df.count()
print("First count:", time.time() - start)
start = time.time()
valid_df.count()
print("Second count:", time.time() - start)
print("Cached:", valid_df.is_cached)
valid_df.unpersist()

First count: 2.2625722885131836
Second count: 0.13441967964172363
Cached: True


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [ ]:
outage_df = readings_df.filter(
    F.col("outage_minutes") > 0
)
outage_df.persist(StorageLevel.MEMORY_AND_DISK)
print("Count:", outage_df.count())
outage_df.groupBy("meter_id").agg(
    F.sum("outage_minutes").alias("total_outage")
).show()
outage_df.unpersist()

Count: 7653
+--------+------------+
|meter_id|total_outage|
+--------+------------+
|   M0063|        3714|
|   M0081|        4745|
|   M0090|        4525|
|   M0116|        5285|
|   M0055|        4467|
|   M0048|        4697|
|   M0102|        4019|
|   M0074|        5602|
|   M0047|        3972|
|   M0039|        4336|
|   M0149|        4388|
|   M0083|        4379|
|   M0056|        6265|
|   M0052|        5101|
|   M0105|        3522|
|   M0013|        4136|
|   M0061|        4018|
|   M0001|        4529|
|   M0121|        5223|
|   M0141|        5122|
+--------+------------+
only showing top 20 rows


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [ ]:
agg_df = readings_df.groupBy(
    "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy")
)
start = time.time()
agg_df.count()
print("Without cache - Run 1:", time.time() - start)
start = time.time()
agg_df.count()
print("Without cache - Run 2:", time.time() - start)
agg_df.cache()
start = time.time()
agg_df.count()
print("With cache - Run 1:", time.time() - start)
start = time.time()
agg_df.count()
print("With cache - Run 2:", time.time() - start)
agg_df.unpersist()

Without cache - Run 1: 1.4017245769500732
Without cache - Run 2: 0.9233677387237549
With cache - Run 1: 5.39677357673645
With cache - Run 2: 2.3456218242645264


DataFrame[meter_id: string, total_energy: double]

G. Partitions and output

In [ ]:
partitioned_df = readings_df.repartition(8)
print("Partitions:",partitioned_df.rdd.getNumPartitions())
partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
).show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11249|
|           1|11249|
|           2|11251|
|           3|11251|
|           4|11250|
|           5|11250|
|           6|11250|
|           7|11250|
+------------+-----+



In [ ]:
df12 = readings_df.repartition(12)
coalesce_df = df12.coalesce(3)
repartition_df = df12.repartition(3)
print("Coalesce plan:")
coalesce_df.explain(True)
print("Repartition plan:")
repartition_df.explain(True)

Coalesce plan:
== Parsed Logical Plan ==
Repartition 3, false
+- Repartition 12, true
   +- Relation [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] csv

== Analyzed Logical Plan ==
reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string
Repartition 3, false
+- Repartition 12, true
   +- Relation [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] csv

== Optimized Logical Plan ==
Repartition 3, false
+- Repartition 12, true
   +- Relation [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_mi

In [ ]:
import os

In [ ]:
enriched_df = filtered_df.join(meter_df,"meter_id","inner")\
  .join(substation_df,"substation_id","inner")
enriched_df.repartition(4).write \
    .mode("overwrite") \
    .parquet("energy_enriched_output")
print("Output files:")
for filename in os.listdir("energy_enriched_output"):
    print(filename)
saved_df = spark.read.parquet("energy_enriched_output")
saved_df.show(5)

Output files:
_SUCCESS
part-00001-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet
.part-00003-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet.crc
.part-00000-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet.crc
part-00000-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet
part-00003-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet
part-00002-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet
._SUCCESS.crc
.part-00001-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet.crc
.part-00002-88871a2b-fc75-4e27-a7d1-80a588bc3058-c000.snappy.parquet.crc
+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-----------+-------------------+-----------------+-----------------+------------+-------------+------+-----------+-----------------+
|substation_id|meter_id|reading_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minut

SECTION 03 / INTEGRATED ASSESSMENT

Final Capstone

Energy Consumption & Reliability Report

In [ ]:
combined = readings_df.unionByName(extra_df)
combined = combined.dropDuplicates(["reading_id"])
print("Unique readings:", combined.count())

combined = combined.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
).withColumn(
    "month",
    F.date_format("reading_timestamp", "yyyy-MM")
)
combined = combined.filter(
    F.col("reading_timestamp").isNotNull()
)

enriched = combined.join(
    meter_df,"meter_id","inner"
).join(
    substation_df,"substation_id","inner"
).join(
    tariff_df,"tariff_id","inner"
)

enriched = enriched.withColumn(
    "billable_energy",
    F.when(
        (F.col("reading_status") != "Rejected") &
        F.col("energy_kwh").isNotNull(),
        F.col("energy_kwh")
    ).otherwise(0.0)
)
enriched = enriched.withColumn(
    "energy_charge",
    F.col("billable_energy") *
    F.col("rate_per_kwh_inr")
)
enriched = enriched.withColumn(
    "outage_flag",
    F.when(
        F.col("outage_minutes") > 0, 1
    ).otherwise(0)
)


report_df = enriched.groupBy(
    "region",
    "site_type",
    "month"
).agg(
    F.sum("billable_energy").alias("total_kwh"),

    F.sum("energy_charge").alias("total_energy_charge_inr"),

    F.sum("outage_minutes").alias("total_outage_minutes"),

    F.count("*").alias("total_readings"),

    F.sum("outage_flag").alias("outage_readings")
)

report_df = report_df.withColumn(
    "outage_percentage",
    F.round(
        F.col("outage_readings") * 100.0 /
        F.col("total_readings"),
        2
    )
)

rank_window = Window.partitionBy(
    "region", "month"
).orderBy(
    F.col("total_energy_charge_inr").desc()
)
report_df = report_df.withColumn(
    "site_rank",
    F.dense_rank().over(rank_window)
)

report_df.cache()
print("Total report rows:", report_df.count())
report_df.orderBy(
    "region", "month", "site_rank"
).show(50, truncate=False)
print("Cached:", report_df.is_cached)

report_df.repartition(2).write \
    .mode("overwrite") \
    .parquet("monthly_energy_report")

report_df.unpersist()

print("Generated Parquet files:")
for filename in os.listdir("monthly_energy_report"):
    print(filename)

final_df = spark.read.parquet("monthly_energy_report")
final_df.show(20, truncate=False)


Unique readings: 101300
Total report rows: 120
+-------+---------------+-------+------------------+-----------------------+--------------------+--------------+---------------+-----------------+---------+
|region |site_type      |month  |total_kwh         |total_energy_charge_inr|total_outage_minutes|total_readings|outage_readings|outage_percentage|site_rank|
+-------+---------------+-------+------------------+-----------------------+--------------------+--------------+---------------+-----------------+---------+
|Central|Industrial     |2026-01|24871.22          |226328.102             |4396                |440           |49             |11.14            |1        |
|Central|Commercial     |2026-01|10649.029999999995|87322.04599999999      |5445                |595           |62             |10.42            |2        |
|Central|Residential    |2026-01|814.03            |4636.908000000001      |1869                |234           |20             |8.55             |3        |
|Central|In